In [1]:
!pip install beautifulsoup4 lxml

In [1]:
from pyspark.sql.functions import col, udf, to_date, from_json, explode, round, input_file_name, regexp_extract
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from bs4 import BeautifulSoup
import subprocess
import re
from pyspark.sql import SparkSession

In [2]:
print("Đang quét phiên bản Hadoop...")
try:
    output = subprocess.check_output(["spark-submit", "--version"], stderr=subprocess.STDOUT).decode("utf-8")
    hadoop_match = re.search(r'Hadoop\s+([0-9\.]+)', output)
    hadoop_version = hadoop_match.group(1) if hadoop_match else "3.3.4"
except Exception:
    hadoop_version = "3.3.4"

if hadoop_version.startswith("3.4"):
    packages = f"org.apache.hadoop:hadoop-aws:{hadoop_version},software.amazon.awssdk:bundle:2.21.24"
    provider_class = "org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider"
else:
    packages = f"org.apache.hadoop:hadoop-aws:{hadoop_version},com.amazonaws:aws-java-sdk-bundle:1.12.262"
    provider_class = "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"

print(f"Đang nạp thư viện: {packages}")

Đang quét phiên bản Hadoop...
Đang nạp thư viện: org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262


In [3]:
print("\nKhởi động Spark (JVM)...")
spark = SparkSession.builder \
    .appName("Financial_ETL_Job") \
    .config("spark.jars.packages", packages) \
    .getOrCreate()
print("Nạp thư viện thành công!")


Khởi động Spark (JVM)...
Nạp thư viện thành công!


In [4]:
hadoop_conf = spark._jsc.hadoopConfiguration()

# Thiết lập cấu hình MinIO
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.access.key", "dataNLPmining-lab")
hadoop_conf.set("fs.s3a.secret.key", "dataNLPmining-lab")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.aws.credentials.provider", provider_class)


In [5]:
iterator = hadoop_conf.iterator()
keys_to_fix = {}
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower() # Chuyển về chữ thường để dễ bắt
    
    # Tìm dạng số đi kèm s, m, h, d (Ví dụ: 30s, 1m, 24h, 1d)
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num = int(match.group(1))
        unit = match.group(2)
        
        # Quy đổi tất cả về milliseconds
        if unit == 's': ms_val = num * 1000
        elif unit == 'm': ms_val = num * 60 * 1000
        elif unit == 'h': ms_val = num * 3600 * 1000
        elif unit == 'd': ms_val = num * 86400 * 1000
        
        keys_to_fix[entry.getKey()] = str(ms_val)

for key, new_val in keys_to_fix.items():
    print(f"   🔧 Vá lỗi: {key} (Đổi '{hadoop_conf.get(key)}' thành '{new_val}')")
    hadoop_conf.set(key, new_val)

print("Đã dọn sạch mọi mầm mống lỗi thời gian (s, m, h, d)!")


   🔧 Vá lỗi: fs.s3a.vectored.read.max.merged.size (Đổi '2M' thành '120000')
   🔧 Vá lỗi: yarn.router.subcluster.cleaner.interval.time (Đổi '60s' thành '60000')
   🔧 Vá lỗi: yarn.resourcemanager.delegation.token.remove-scan-interval (Đổi '1h' thành '3600000')
   🔧 Vá lỗi: yarn.router.interceptor.user-thread-pool.keep-alive-time (Đổi '30s' thành '30000')
   🔧 Vá lỗi: fs.s3a.connection.establish.timeout (Đổi '30s' thành '30000')
   🔧 Vá lỗi: fs.s3a.multipart.purge.age (Đổi '24h' thành '86400000')
   🔧 Vá lỗi: yarn.federation.state-store.heartbeat.initial-delay (Đổi '30s' thành '30000')
   🔧 Vá lỗi: fs.s3a.threads.keepalivetime (Đổi '60s' thành '60000')
   🔧 Vá lỗi: hadoop.security.groups.shell.command.timeout (Đổi '0s' thành '0')
   🔧 Vá lỗi: hadoop.service.shutdown.timeout (Đổi '30s' thành '30000')
   🔧 Vá lỗi: yarn.federation.state-store.clean-up-retry-sleep-time (Đổi '1s' thành '1000')
   🔧 Vá lỗi: fs.s3a.multipart.threshold (Đổi '128M' thành '7680000')
   🔧 Vá lỗi: yarn.router.subclus

In [7]:
try:
    uri = spark._jvm.java.net.URI("s3a://raw-financial-data/")
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(uri, hadoop_conf)
    Path = spark._jvm.org.apache.hadoop.fs.Path
    
    folder_path = Path("s3a://raw-financial-data/raw_zone/news_data/")
    statuses = fs.listStatus(folder_path)
    
    print("\nKẾT NỐI THÀNH CÔNG, DANH SÁCH FILE TRÊN MINIO:")
    print("=" * 70)
    for status in statuses:
        print(f"📄 Tên file: {status.getPath().toString()}")
    print("=" * 70)
        
except Exception as e:
    print(f"❌ LỖI KẾT NỐI: {e}")


KẾT NỐI THÀNH CÔNG, DANH SÁCH FILE TRÊN MINIO:
📄 Tên file: s3a://raw-financial-data/raw_zone/news_data/2026


In [8]:
def clean_html(raw_text):
    if not raw_text:
        return ""
    return BeautifulSoup(raw_text, "lxml").text.strip()

In [9]:
clean_html_udf = udf(clean_html, StringType())

content_schema = StructType([
    StructField("id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("pubDate", StringType(), True)
])
root_schema = StructType([
    StructField("id", StringType(), True),
    StructField("content", content_schema, True)
])

In [13]:
df_raw_news = spark.read.option("wholetext", "true").text("s3a://raw-financial-data/raw_zone/news_data/*/*/*/*.json")

# KỸ THUẬT MỚI: Trích xuất Ticker từ đường dẫn file (VD: s3a://.../AAPL.json -> AAPL)
df_raw_news = df_raw_news.withColumn("Ticker", regexp_extract(input_file_name(), r'([^/]+)\.json$', 1))

# Parse và Flatten
df_parsed = df_raw_news.withColumn("json_array", from_json(col("value"), ArrayType(root_schema)))
df_exploded = df_parsed.select(col("Ticker"), explode(col("json_array")).alias("article"))

df_news_flat = df_exploded.select(
    col("Ticker"),
    col("article.content.id").alias("uuid"),
    col("article.content.title").alias("title"),
    col("article.content.summary").alias("content_raw"),
    col("article.content.pubDate").alias("pubDate")
)

df_news_clean = df_news_flat.dropna(subset=["uuid", "pubDate"]) \
    .withColumn("CleanedText", clean_html_udf(col("content_raw"))) \
    .withColumn("Date", to_date(col("pubDate"))) \
    .select("Date", "Ticker", "uuid", "title", "CleanedText")

df_news_clean.show(10, truncate=50)

+----------+------+------------------------------------+--------------------------------------------------+--------------------------------------------------+
|      Date|Ticker|                                uuid|                                             title|                                       CleanedText|
+----------+------+------------------------------------+--------------------------------------------------+--------------------------------------------------+
|2026-06-01|  MSFT|47d6de09-8456-4330-9121-a92c931adb53|Nvidia partners with Microsoft on new RTX Spark...|Nvidia CEO Jensen Huang introduces the new RTX ...|
|2026-06-01|  MSFT|48e2edc1-8e96-434b-8bdf-fb0f28451d4c|The value Nvidia's new chips will add to Window...|Nvidia stock (NVDA) cruises higher to start off...|
|2026-06-01|  MSFT|da795e64-6fad-4c7a-9bec-9732d692f2f4|Meta, Alphabet, Amazon, and Microsoft are getti...|Debt is becoming a big tool in the Big Tech ars...|
|2026-06-02|  MSFT|7b520322-5eda-3b78-b880-13b

In [14]:
df_raw_price = spark.read.option("multiline", "true").json("s3a://raw-financial-data/raw_zone/market_data/*/*/*/*.json")

# Lấy Ticker từ tên file giá cổ phiếu
df_price_clean = df_raw_price.withColumn("Ticker", regexp_extract(input_file_name(), r'([^/]+)\.json$', 1)) \
    .withColumn("Date", to_date(col("Date"))) \
    .withColumn("DailyReturn", round((col("Close") - col("Open")) / col("Open"), 4)) \
    .select("Date", "Ticker", "Open", "Close", "DailyReturn")

df_price_clean.show(10, truncate=50)

+----------+------+------------------+------------------+-----------+
|      Date|Ticker|              Open|             Close|DailyReturn|
+----------+------+------------------+------------------+-----------+
|2025-06-26|    FB| 39.61882845277748|39.391815185546875|    -0.0057|
|2025-06-27|    FB|39.628700091438006| 39.72641372680664|     0.0025|
|2025-06-30|    FB| 39.92480728489936| 39.94849395751953|     6.0E-4|
|2025-07-01|    FB| 39.83795135601928|39.869537353515625|     8.0E-4|
|2025-07-02|    FB|39.787614354873526|  39.7274055480957|    -0.0015|
|2025-07-03|    FB| 39.95441331882837|39.987972259521484|     8.0E-4|
|2025-07-07|    FB| 40.04324694037468| 40.02350616455078|    -5.0E-4|
|2025-07-08|    FB| 39.95441722156564| 39.88532638549805|    -0.0017|
|2025-07-09|    FB| 39.97415465949547| 39.91986846923828|    -0.0014|
|2025-07-10|    FB| 39.96428813744811|39.950469970703125|    -3.0E-4|
+----------+------+------------------+------------------+-----------+
only showing top 10 

In [17]:
df_final = df_news_clean.join(df_price_clean, on=["Date", "Ticker"], how="inner")

df_final.show(10, truncate=30)

+----------+------+------------------------------+------------------------------+------------------------------+------------------+-----------------+-----------+
|      Date|Ticker|                          uuid|                         title|                   CleanedText|              Open|            Close|DailyReturn|
+----------+------+------------------------------+------------------------------+------------------------------+------------------+-----------------+-----------+
|2026-06-01|  MSFT|47d6de09-8456-4330-9121-a92...|Nvidia partners with Micros...|Nvidia CEO Jensen Huang int...|465.05999755859375|460.5199890136719|    -0.0098|
|2026-06-01|  MSFT|48e2edc1-8e96-434b-8bdf-fb0...|The value Nvidia's new chip...|Nvidia stock (NVDA) cruises...|465.05999755859375|460.5199890136719|    -0.0098|
|2026-06-01|  MSFT|da795e64-6fad-4c7a-9bec-973...|Meta, Alphabet, Amazon, and...|Debt is becoming a big tool...|465.05999755859375|460.5199890136719|    -0.0098|
|2026-06-01|  MSFT|d0134165-

In [18]:
output_path = "s3a://raw-financial-data/silver_zone/news_price_merged.parquet"
df_final.write.mode("overwrite").parquet(output_path)
print(f"🎉 Đã lưu Parquet thành công tại: {output_path}")

🎉 Đã lưu Parquet thành công tại: s3a://raw-financial-data/silver_zone/news_price_merged.parquet
